# Practical Exercise 4: Knowledge Engineering
## A Knowledge-Based Autonomous Trailer Agent

In Practical Exercise 3 our autonomous tractor and trailer used **Uniform Cost Search (UCS)** to find the shortest route from Nairobi to Kisumu over a small road network. That agent was a *problem-solving agent*: we handed it a fixed graph and it searched it.

In this practical exercise we turn the trailer into a **knowledge-based agent**. Instead of a fixed graph hidden inside the search code, the trailer now keeps an explicit **knowledge base (KB)**: a store of facts about its world (towns, roads, distances, road conditions, where it is, where it must go) and a set of **rules** it uses to reason about those facts. The trailer can be *told* new facts (for example, "the Nakuru–Mau Summit road is closed"), it can be *asked* questions ("which towns can I reach?"), and it uses what it knows to *decide* which route to drive.

UCS is still here. It is now one part of the agent: the KB decides which roads can be used, and UCS finds the cheapest route over those roads only.

| | Conventional route finder (Practical Exercise 3) | Knowledge-based agent (Practical Exercise 4) |
|---|---|---|
| World model | A graph fixed in the code | Facts stored in a knowledge base |
| Road conditions | Not represented, so every road is assumed usable | Stored as facts; rules decide whether a road is usable |
| Change in the world | The programmer must edit the graph | The agent is told a new fact and replans |
| Questions it can answer | "What is the shortest path?" | Shortest path, plus: Is this road open? What can I reach? Why did I choose this route? |
| Decisions | Computed once | Recomputed from current knowledge |

We follow the ten knowledge engineering steps used in the lecturer's reference notebook (*Smart Traffic Light Navigation Agent*), and we use the same simple techniques: Python dictionaries keyed by predicate name, a rules dictionary with a cautious default for unknown cases, small functions that add facts, show the KB and answer queries, and a list of test cases for debugging.

**Core implementation:** Steps 1–10.
**Optional extension:** the section after Step 9, where the trailer drives the route one road at a time and replans when it observes a closure along the way.

### Setup

We need only two tools from the Python standard library, both taken over from Practical Exercise 3:
- `heapq` provides the priority queue (the *frontier*) that UCS uses to always explore the cheapest route next.
- `count` provides a tie-break number so that two routes with equal cost leave the queue in a predictable order.

In [1]:
# heapq gives us a priority queue: UCS uses it to pick the cheapest waiting route.
import heapq

# count() produces 0, 1, 2, ... and is used to break ties between equal-cost queue entries.
from itertools import count

## Step 1: Identify the Task

**Task:** An autonomous tractor and trailer must deliver cargo from its current town to a destination town, using only roads it knows are usable, and choosing the shortest such route.

| Element | Description |
|---|---|
| **Environment** | A town-level road network in Kenya: 10 towns joined by 12 two-way roads (from Practical Exercise 3). Roads can change condition (open, wet, closed, flooded, weight restricted). |
| **Agent** | The autonomous trailer. It perceives its location and road conditions, stores what it knows in a KB, reasons over that knowledge and chooses its movements. |
| **Available information** | The towns, the roads between them and their distances, the condition of each road, the trailer's current location and its delivery destination. |
| **Objective** | Reach the destination with the cargo intact, using only usable roads, while minimising the total distance driven. |
| **Decisions** | Is a road usable? Which towns can be reached from here? Is the destination reachable at all? Which route is cheapest? Which town should the trailer drive to next? Should it replan after a new observation? |

**Why this is a knowledge-based problem.** The right decision depends on what the agent *knows* at the moment it decides. The same road network gives a different best route depending on which roads are closed. If the agent's knowledge is kept separate from its reasoning, we can change the facts (a closure, a reopening) without touching the code, and the agent will reach a new, correct conclusion by applying the same rules.

**Simplifying assumptions (carried over from Practical Exercise 3).** At town level we treat the world as fully observable, deterministic, static between observations, and discrete. Every road is two-way with equal distance in both directions. Distances are approximate published values rounded to whole kilometres. The optimal route is optimal *for this graph only*, not for every road in Kenya.

## Step 2: Assemble Relevant Knowledge

Before we can write anything in Python we must decide what the trailer needs to know. Knowledge falls into two groups:

**1. Knowledge about the environment (facts)** describes the particular world the trailer is in. It can change.

| What the agent needs to know | Source |
|---|---|
| Which towns exist | Practical Exercise 3 road network |
| Which roads connect which towns, and their distances | Practical Exercise 3 road network (table below) |
| The current condition of each road | Initially all *Open* (Practical Exercise 3 assumed this); later updated by observations |
| Where the trailer is now | The problem instance (Step 6), later updated as it moves |
| Where the trailer must deliver its cargo | The problem instance (Step 6) |

**2. Knowledge about how to reason (rules)** is general and holds in every situation:

- A road in a usable condition (Open, Wet, Under Repair) may be travelled. A road in an unusable condition (Closed, Flooded, Weight Restricted) must not be used.
- A road condition the agent has no rule for is treated as unusable, to be safe.
- A town is directly reachable if a usable road leads to it from the current town. It is reachable at all if a chain of usable roads leads to it.
- When several usable routes exist, the trailer should take the one with the lowest total distance.
- Movement restriction: the trailer is in exactly one town at a time and moves only along usable roads.

Keeping these groups apart is the heart of knowledge engineering. The rules are written once. The facts are what we change when the world changes.

**Data sources:** No APIs or web scraping were used. The road distances are the ones assembled for Practical Exercise 3 (approximate published distances, rounded to whole kilometres).

The road data below is copied unchanged from the Practical Exercise 3 notebook.

In [2]:
# Environment knowledge carried over unchanged from Practical Exercise 3.
# Each tuple reads: (first town, second town, road distance in km).
roadConnections = [
    ("Nairobi", "Nakuru", 161),
    ("Nairobi", "Nyeri", 151),
    ("Nyeri", "Nyahururu", 104),
    ("Nyahururu", "Nakuru", 64),
    ("Nakuru", "Mau Summit", 62),
    ("Mau Summit", "Kericho", 50),
    ("Kericho", "Kisumu", 84),
    ("Nairobi", "Narok", 141),
    ("Narok", "Bomet", 81),
    ("Bomet", "Kericho", 77),
    ("Bomet", "Kisii", 83),
    ("Kisii", "Kisumu", 114),
]

print(f"{'Starting Town':<15}{'Destination':<15}{'Distance':>10}")
print("-" * 40)
for firstTown, secondTown, roadDistance in roadConnections:
    print(f"{firstTown:<15}{secondTown:<15}{roadDistance:>7} km")

townsMentioned = set()
for firstTown, secondTown, roadDistance in roadConnections:
    townsMentioned.add(firstTown)
    townsMentioned.add(secondTown)

print(f"\n{len(townsMentioned)} towns and {len(roadConnections)} road connections.")

Starting Town  Destination      Distance
----------------------------------------
Nairobi        Nakuru             161 km
Nairobi        Nyeri              151 km
Nyeri          Nyahururu          104 km
Nyahururu      Nakuru              64 km
Nakuru         Mau Summit          62 km
Mau Summit     Kericho             50 km
Kericho        Kisumu              84 km
Nairobi        Narok              141 km
Narok          Bomet               81 km
Bomet          Kericho             77 km
Bomet          Kisii               83 km
Kisii          Kisumu             114 km

10 towns and 12 road connections.


## Step 3: Define the Vocabulary

The vocabulary is the set of **predicates** (relationships that can be true or false) and **constants** (names of particular things) we use to describe the world. As in the lecturer's notebook, predicates are represented in Python as **dictionary keys**, and the facts for each predicate are stored under its key.

**Stored predicates (facts we are told):**

| Predicate | Meaning | Python representation |
|---|---|---|
| `Town(x)` | x is a town in the trailer's world | `knowledgeBase["Town"]` is a set of town names |
| `Road(x, y, d)` | a road leads from x to y and is d km long | `knowledgeBase["Road"][(x, y)] = d` |
| `RoadCondition(x, y, c)` | the road from x to y is currently in condition c | `knowledgeBase["RoadCondition"][(x, y)] = c` |
| `At(Trailer, x)` | the trailer is currently in town x | `knowledgeBase["At"]["Trailer"] = x` |
| `Destination(Trailer, x)` | the trailer must deliver its cargo to x | `knowledgeBase["Destination"]["Trailer"] = x` |

**Derived predicates (worked out by rules in Step 4, never stored):**

| Predicate | Meaning |
|---|---|
| `CanUse(x, y)` | the road from x to y exists and its condition allows the trailer to use it |
| `RoadOpen(x, y)` | the same as `CanUse(x, y)` |
| `RoadClosed(x, y)` | the road from x to y exists but the trailer may not use it |
| `Reachable(x, y)` | a chain of usable roads leads from x to y |

`RoadOpen` and `RoadClosed` are not stored separately because they follow from `RoadCondition`. If we stored them as well, the facts could contradict each other (a road marked both open and closed). Working them out from a single source avoids that.

**Constants:**
- The agent: `Trailer`.
- Towns: `Nairobi`, `Nakuru`, `Nyeri`, `Nyahururu`, `Mau Summit`, `Kericho`, `Kisumu`, `Narok`, `Bomet`, `Kisii`.
- Road conditions: `Open`, `Wet`, `Under Repair`, `Closed`, `Flooded`, `Weight Restricted` (defined with their rules in Step 4).

Each two-way road is stored as **two directed facts**, `Road(x, y, d)` and `Road(y, x, d)`. This is the same way Practical Exercise 3's `buildGraph()` added one arc in each direction.

In [3]:
# The knowledge base. Each key is the name of a predicate and each value holds
# the facts we currently know for that predicate.
knowledgeBase = {
    "Town": set(),         # Town(x)
    "Road": {},            # Road(x, y, d): (x, y) -> d
    "RoadCondition": {},   # RoadCondition(x, y, c): (x, y) -> c
    "At": {},              # At(Trailer, x): "Trailer" -> x
    "Destination": {},     # Destination(Trailer, x): "Trailer" -> x
}

# The constant that names our agent.
trailer = "Trailer"

Constants only work if every part of the program writes them the same way. `"nakuru"`, `" Nakuru "` and `"Nakuru"` should all mean the same town. The lecturer's notebook handled this with `.capitalize()`. We use `.title()` instead, because some of our names are two words (`Mau Summit`). Every function that reads or writes the KB passes names through `normalizeName()` first.

In [4]:
def normalizeName(name):
    """Write a constant in the standard form used by the KB, e.g. '  mau summit ' -> 'Mau Summit'."""
    return name.strip().title()


print(normalizeName("nakuru"))
print(normalizeName("  mau summit "))
print(normalizeName("WEIGHT RESTRICTED"))

Nakuru
Mau Summit
Weight Restricted


## Step 4: Encode General Knowledge about the Domain

Now we write the **rules**: general knowledge that holds no matter which roads exist or where the trailer is.

### 4.1 Road condition rules

This is the direct counterpart of the lecturer's `rules` dictionary, which mapped each traffic light colour to an action. Here each **road condition** maps to whether the trailer may use the road, plus the advice the agent should follow. The rule *"a road must be suitable for the trailer before it can be considered"* is captured here too: a `Weight Restricted` road exists and is not closed, but it is not suitable for a loaded trailer.

As in the reference, an unknown input gets a **cautious default** (the lecturer's "Unknown signal: proceed with caution or stop"). A condition the agent has never heard of is treated as unusable.

In [5]:
# General rules about road conditions: condition -> whether the trailer may use the road, and advice.
roadConditionRules = {
    "Open":              {"usable": True,  "advice": "Usable: the road is open to the trailer."},
    "Wet":               {"usable": True,  "advice": "Usable: reduce speed, the surface is slippery."},
    "Under Repair":      {"usable": True,  "advice": "Usable: expect delays from roadworks."},
    "Closed":            {"usable": False, "advice": "Not usable: the road is closed to all traffic."},
    "Flooded":           {"usable": False, "advice": "Not usable: the trailer cannot safely cross flood water."},
    "Weight Restricted": {"usable": False, "advice": "Not usable: the loaded trailer exceeds the weight limit."},
}

# Cautious default for any condition we have no rule for.
unknownConditionRule = {"usable": False, "advice": "Unknown condition: treat the road as unusable until confirmed safe."}


def getConditionRule(condition):
    """Look up the rule for a road condition, falling back to the cautious default (like rules.get in the reference)."""
    return roadConditionRules.get(normalizeName(condition), unknownConditionRule)

### 4.2 Rules for using roads

These functions do not store anything. Each one **works out** a derived predicate from the stored facts:

- **CanUse rule:** `Road(x, y, d)` ∧ `RoadCondition(x, y, c)` ∧ the rule for *c* says usable ⇒ `CanUse(x, y)`
- **RoadOpen** is the same as `CanUse`.
- **RoadClosed rule:** `Road(x, y, d)` ∧ ¬`CanUse(x, y)` ⇒ `RoadClosed(x, y)`

If a road has no recorded condition, the agent treats it as `Unknown`, so the cautious default applies.

In [6]:
def roadExists(fromTown, toTown):
    """True if the fact Road(fromTown, toTown, d) is in the KB for some distance d."""
    return (normalizeName(fromTown), normalizeName(toTown)) in knowledgeBase["Road"]


def canUseRoad(fromTown, toTown):
    """
    CanUse rule: the road must exist AND the rule for its current condition must say it is usable.
    """
    fromTown = normalizeName(fromTown)
    toTown = normalizeName(toTown)

    if not roadExists(fromTown, toTown):
        return False

    condition = knowledgeBase["RoadCondition"].get((fromTown, toTown), "Unknown")
    return getConditionRule(condition)["usable"]


def isRoadOpen(fromTown, toTown):
    """RoadOpen(x, y) means the same as CanUse(x, y)."""
    return canUseRoad(fromTown, toTown)


def isRoadClosed(fromTown, toTown):
    """RoadClosed rule: a known road that the trailer is not allowed to use."""
    return roadExists(fromTown, toTown) and not canUseRoad(fromTown, toTown)

### 4.3 Rules for reachability

- **Direct reachability:** `CanUse(x, y)` ⇒ y is directly reachable from x. `getUsableNeighbours()` lists these towns with their distances, ordered as in Practical Exercise 3 (shortest road first, then alphabetically).
- **Reachability (two rules):**
  1. `CanUse(x, y)` ⇒ `Reachable(x, y)`
  2. `Reachable(x, y)` ∧ `CanUse(y, z)` ⇒ `Reachable(x, z)`

`findReachableTowns()` applies these rules by **forward chaining**. It starts from what is directly reachable, then keeps applying rule 2 to each newly inferred town until nothing new can be inferred. This differs from the route search in Step 8. It only answers *whether* a town can be reached, not *how cheaply*.

In [7]:
def getUsableNeighbours(town):
    """Return (neighbour, distance) for every road out of town that the CanUse rule allows."""
    town = normalizeName(town)
    usableNeighbours = []

    for (fromTown, toTown), roadDistance in knowledgeBase["Road"].items():
        if fromTown == town and canUseRoad(fromTown, toTown):
            usableNeighbours.append((toTown, roadDistance))

    # Same ordering as Practical Exercise 3's buildGraph(): shortest road first, then by name.
    usableNeighbours.sort(key=lambda connection: (connection[1], connection[0]))
    return usableNeighbours


def findReachableTowns(startTown):
    """
    Forward chaining over the two Reachable rules.
    townsToExpand holds towns we have inferred as reachable but whose roads we have not yet checked.
    """
    startTown = normalizeName(startTown)
    reachableTowns = set()
    townsToExpand = [startTown]

    while len(townsToExpand) > 0:
        town = townsToExpand.pop()
        for neighbourTown, roadDistance in getUsableNeighbours(town):
            # Only record a town the first time we infer it; this also stops us going round cycles.
            if neighbourTown != startTown and neighbourTown not in reachableTowns:
                reachableTowns.add(neighbourTown)
                townsToExpand.append(neighbourTown)

    return reachableTowns

### 4.4 Rule for choosing a route

**Lowest-cost rule:** *if several usable routes lead to the destination, choose the one with the lowest total distance.* This rule is carried out by the Uniform Cost Search from Practical Exercise 3, which we connect to the KB in Step 8.

The KB is still empty at this point, so these rules have nothing to reason about yet. We use them once the facts are added in Step 5.

## Step 5: Add New Facts to the Knowledge Base

We need functions that **tell** the KB new facts, in the spirit of the lecturer's `add_new_fact()` and `show_kb()`:

- `addTown(x)` adds `Town(x)`.
- `addRoad(x, y, d)` adds `Road(x, y, d)` and `Road(y, x, d)` (two-way, as in Practical Exercise 3), each with an initial condition of `Open`. A road can only join towns, so it also adds `Town(x)` and `Town(y)`. Practical Exercise 3's `buildGraph()` added towns in the same way.
- `setRoadCondition(x, y, c)` **updates** the condition of an existing road in both directions. It refuses to invent a road the KB does not know about.
- `showKnowledgeBase()` prints every fact grouped by predicate.

In [8]:
def addTown(town):
    """Tell the KB the fact Town(town). Returns the normalised name."""
    town = normalizeName(town)
    knowledgeBase["Town"].add(town)
    return town


def addRoad(firstTown, secondTown, roadDistance, condition="Open"):
    """Tell the KB about a two-way road: Road(x, y, d), Road(y, x, d) and their conditions."""
    firstTown = addTown(firstTown)
    secondTown = addTown(secondTown)
    condition = normalizeName(condition)

    for fromTown, toTown in [(firstTown, secondTown), (secondTown, firstTown)]:
        knowledgeBase["Road"][(fromTown, toTown)] = roadDistance
        knowledgeBase["RoadCondition"][(fromTown, toTown)] = condition


def setRoadCondition(firstTown, secondTown, condition):
    """Update RoadCondition for both directions of an existing road. Returns True if the KB changed."""
    firstTown = normalizeName(firstTown)
    secondTown = normalizeName(secondTown)
    condition = normalizeName(condition)

    if not roadExists(firstTown, secondTown):
        print(f"  Rejected: the KB has no road between {firstTown} and {secondTown}.")
        return False

    knowledgeBase["RoadCondition"][(firstTown, secondTown)] = condition
    knowledgeBase["RoadCondition"][(secondTown, firstTown)] = condition
    return True

In [9]:
def showKnowledgeBase():
    """Print every fact currently in the KB, grouped by predicate (compare the lecturer's show_kb)."""
    print("\n--- Current Knowledge Base ---")
    print(f"Town facts ({len(knowledgeBase['Town'])}): {', '.join(sorted(knowledgeBase['Town']))}")

    print(f"\nRoad facts ({len(knowledgeBase['Road'])} directed; each two-way road is shown once):")
    for (fromTown, toTown), roadDistance in sorted(knowledgeBase["Road"].items()):
        if fromTown < toTown:
            condition = knowledgeBase["RoadCondition"].get((fromTown, toTown), "Unknown")
            roadFact = f"Road({fromTown}, {toTown}, {roadDistance})"
            print(f"  {roadFact:<34} RoadCondition: {condition}")

    print("\nAt facts:")
    for agentName, town in knowledgeBase["At"].items():
        print(f"  At({agentName}, {town})")
    if len(knowledgeBase["At"]) == 0:
        print("  (none yet)")

    print("Destination facts:")
    for agentName, town in knowledgeBase["Destination"].items():
        print(f"  Destination({agentName}, {town})")
    if len(knowledgeBase["Destination"]) == 0:
        print("  (none yet)")
    print("------------------------------")

The lecturer's notebook also showed facts being added through `input()`. We include the same idea below, but leave the call commented out so the notebook can run from top to bottom without waiting for typing.

In [10]:
def addRoadFromUser():
    """Interactive version of addRoad, like the lecturer's add_fact()."""
    firstTown = input("First town: ")
    secondTown = input("Second town: ")
    roadDistance = int(input("Distance in km: "))
    addRoad(firstTown, secondTown, roadDistance)
    print(f"New fact added: Road({normalizeName(firstTown)}, {normalizeName(secondTown)}, {roadDistance}) in both directions")


# addRoadFromUser()   # uncomment to add a road interactively

### 5.1 Populate the KB with the Practical Exercise 3 road network

Every road in `roadConnections` becomes two `Road` facts and two `RoadCondition` facts. We then check that the KB holds exactly what Practical Exercise 3 described: 10 towns, and 12 two-way roads stored as 24 directed facts.

In [11]:
for firstTown, secondTown, roadDistance in roadConnections:
    addRoad(firstTown, secondTown, roadDistance)

showKnowledgeBase()

assert len(knowledgeBase["Town"]) == 10, "Expected 10 towns from Practical Exercise 3."
assert len(knowledgeBase["Road"]) == 24, "Expected 12 two-way roads = 24 directed Road facts."
assert len(knowledgeBase["RoadCondition"]) == 24, "Every Road fact should have a RoadCondition fact."
print("\nCheck passed: the KB holds the 10 towns and 12 two-way roads from Practical Exercise 3.")


--- Current Knowledge Base ---
Town facts (10): Bomet, Kericho, Kisii, Kisumu, Mau Summit, Nairobi, Nakuru, Narok, Nyahururu, Nyeri

Road facts (24 directed; each two-way road is shown once):
  Road(Bomet, Kericho, 77)           RoadCondition: Open
  Road(Bomet, Kisii, 83)             RoadCondition: Open
  Road(Bomet, Narok, 81)             RoadCondition: Open
  Road(Kericho, Kisumu, 84)          RoadCondition: Open
  Road(Kericho, Mau Summit, 50)      RoadCondition: Open
  Road(Kisii, Kisumu, 114)           RoadCondition: Open
  Road(Mau Summit, Nakuru, 62)       RoadCondition: Open
  Road(Nairobi, Nakuru, 161)         RoadCondition: Open
  Road(Nairobi, Narok, 141)          RoadCondition: Open
  Road(Nairobi, Nyeri, 151)          RoadCondition: Open
  Road(Nakuru, Nyahururu, 64)        RoadCondition: Open
  Road(Nyahururu, Nyeri, 104)        RoadCondition: Open

At facts:
  (none yet)
Destination facts:
  (none yet)
------------------------------

Check passed: the KB holds the 10 t

### 5.2 Retrieving and updating facts

Facts are **retrieved** by looking them up by predicate and arguments. They are **updated** by telling the KB a new value. In the example below, rain is reported on the Narok–Bomet road. We update its condition and let the rules decide what that means. Then the rain clears and we restore it. Note that the agent's conclusion (*usable or not*) is not stored anywhere. It is recomputed from the fact and the rule each time.

In [12]:
# Retrieving facts
print("Road(Nairobi, Nakuru, d): d =", knowledgeBase["Road"][("Nairobi", "Nakuru")], "km")
print("Road(Nakuru, Nairobi, d): d =", knowledgeBase["Road"][("Nakuru", "Nairobi")], "km  (the reverse direction)")
print("RoadCondition(Nairobi, Nakuru):", knowledgeBase["RoadCondition"][("Nairobi", "Nakuru")])
print("Town(Bomet)?", "Bomet" in knowledgeBase["Town"])

# Updating a fact: rain is reported on Narok-Bomet.
print("\nUpdate: the Narok-Bomet road is reported wet.")
setRoadCondition("Narok", "Bomet", "Wet")
print("  RoadCondition(Narok, Bomet):", knowledgeBase["RoadCondition"][("Narok", "Bomet")])
print("  Rule applied:", getConditionRule("Wet")["advice"])
print("  CanUse(Narok, Bomet)?", canUseRoad("Narok", "Bomet"))

# The rain clears, so the original fact is restored.
print("\nUpdate: the Narok-Bomet road has dried.")
setRoadCondition("Narok", "Bomet", "Open")
print("  RoadCondition(Narok, Bomet):", knowledgeBase["RoadCondition"][("Narok", "Bomet")])

Road(Nairobi, Nakuru, d): d = 161 km
Road(Nakuru, Nairobi, d): d = 161 km  (the reverse direction)
RoadCondition(Nairobi, Nakuru): Open
Town(Bomet)? True

Update: the Narok-Bomet road is reported wet.
  RoadCondition(Narok, Bomet): Wet
  Rule applied: Usable: reduce speed, the surface is slippery.
  CanUse(Narok, Bomet)? True

Update: the Narok-Bomet road has dried.
  RoadCondition(Narok, Bomet): Open


## Step 6: Encode the Specific Problem Instance

Steps 1–5 describe the world in general. Now we describe **the particular situation** the agent is in:

- `At(Trailer, Nairobi)`: the trailer is in Nairobi.
- `Destination(Trailer, Kisumu)`: the cargo must go to Kisumu.
- The road network is the Practical Exercise 3 network with every road open (already in the KB from Step 5).

The setter functions only accept towns the KB knows about. This keeps the problem instance consistent with the rest of the knowledge. `At` holds a single value per agent because the trailer can only be in one town at a time, so a new location replaces the old one.

In the lecturer's notebook the current observation came from `input()`. On a real trailer it would come from GPS and the delivery order. We set the values directly so the notebook runs unattended.

In [13]:
def setTrailerLocation(town):
    """Tell the KB At(Trailer, town). Only known towns are accepted."""
    town = normalizeName(town)
    if town not in knowledgeBase["Town"]:
        print(f"  Rejected: '{town}' is not a known town, so At({trailer}, {town}) was not added.")
        return False
    knowledgeBase["At"][trailer] = town
    return True


def setDestination(town):
    """Tell the KB Destination(Trailer, town). Only known towns are accepted."""
    town = normalizeName(town)
    if town not in knowledgeBase["Town"]:
        print(f"  Rejected: '{town}' is not a known town, so Destination({trailer}, {town}) was not added.")
        return False
    knowledgeBase["Destination"][trailer] = town
    return True

In [14]:
# The specific problem instance (same as Practical Exercise 3's initialState and goalState).
# startingLocation = input("Where is the trailer now? ")   # interactive alternative
startingLocation = "Nairobi"
deliveryDestination = "Kisumu"

setTrailerLocation(startingLocation)
setDestination(deliveryDestination)

showKnowledgeBase()


--- Current Knowledge Base ---
Town facts (10): Bomet, Kericho, Kisii, Kisumu, Mau Summit, Nairobi, Nakuru, Narok, Nyahururu, Nyeri

Road facts (24 directed; each two-way road is shown once):
  Road(Bomet, Kericho, 77)           RoadCondition: Open
  Road(Bomet, Kisii, 83)             RoadCondition: Open
  Road(Bomet, Narok, 81)             RoadCondition: Open
  Road(Kericho, Kisumu, 84)          RoadCondition: Open
  Road(Kericho, Mau Summit, 50)      RoadCondition: Open
  Road(Kisii, Kisumu, 114)           RoadCondition: Open
  Road(Mau Summit, Nakuru, 62)       RoadCondition: Open
  Road(Nairobi, Nakuru, 161)         RoadCondition: Open
  Road(Nairobi, Narok, 141)          RoadCondition: Open
  Road(Nairobi, Nyeri, 151)          RoadCondition: Open
  Road(Nakuru, Nyahururu, 64)        RoadCondition: Open
  Road(Nyahururu, Nyeri, 104)        RoadCondition: Open

At facts:
  At(Trailer, Nairobi)
Destination facts:
  Destination(Trailer, Kisumu)
------------------------------


## Step 7: Pose Queries and Get Answers

We now **ask** the KB questions. Each query function answers in one of two ways:

- **Fact lookup:** the answer is stored directly (e.g. the distance of a road).
- **Inference:** the answer is not stored anywhere. A rule works it out from stored facts (e.g. whether a road is open, which towns are reachable).

As in the reference, a question about something the KB does not know gets a clear "unknown" answer instead of a crash.

In [15]:
def listTowns():
    """Which towns exist? (fact lookup)"""
    return sorted(knowledgeBase["Town"])


def isKnownTown(town):
    """Is Town(town) in the KB? (fact lookup)"""
    return normalizeName(town) in knowledgeBase["Town"]


def getConnectedTowns(town):
    """Which towns does a Road fact join to this town, whatever the road's condition? (fact lookup)"""
    town = normalizeName(town)
    connectedTowns = []
    for (fromTown, toTown), roadDistance in knowledgeBase["Road"].items():
        if fromTown == town:
            connectedTowns.append((toTown, roadDistance))
    connectedTowns.sort(key=lambda connection: (connection[1], connection[0]))
    return connectedTowns


def getRoadDistance(fromTown, toTown):
    """Distance d from Road(x, y, d), or None if no such road is known. (fact lookup)"""
    return knowledgeBase["Road"].get((normalizeName(fromTown), normalizeName(toTown)))


def describeRoadStatus(fromTown, toTown):
    """Is the road open? Answer with its condition and the advice from the matching rule. (inference)"""
    fromTown = normalizeName(fromTown)
    toTown = normalizeName(toTown)
    if not roadExists(fromTown, toTown):
        return f"Unknown road: the KB has no Road fact from {fromTown} to {toTown}."
    condition = knowledgeBase["RoadCondition"].get((fromTown, toTown), "Unknown")
    return f"{condition}. {getConditionRule(condition)['advice']}"

In [16]:
def getCurrentLocation():
    """Where is the trailer? Looks up At(Trailer, x)."""
    return knowledgeBase["At"].get(trailer)


def getDestination():
    """Where must the trailer go? Looks up Destination(Trailer, x)."""
    return knowledgeBase["Destination"].get(trailer)


def canReachDestination():
    """Does Reachable(current location, destination) hold? (inference)"""
    currentLocation = getCurrentLocation()
    destination = getDestination()
    if currentLocation is None or destination is None:
        return False
    if currentLocation == destination:
        return True
    return destination in findReachableTowns(currentLocation)


def listUnavailableRoads():
    """Every road for which RoadClosed(x, y) can be inferred, each two-way road listed once. (inference)"""
    unavailableRoads = []
    for fromTown, toTown in knowledgeBase["Road"]:
        if fromTown < toTown and isRoadClosed(fromTown, toTown):
            condition = knowledgeBase["RoadCondition"].get((fromTown, toTown), "Unknown")
            unavailableRoads.append((fromTown, toTown, condition))
    return sorted(unavailableRoads)


def printAnswer(question, answer, howAnswered):
    """Print a question, the agent's answer and how the answer was obtained."""
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"   [{howAnswered}]\n")

In [17]:
printAnswer("Which towns exist?", ", ".join(listTowns()), "fact lookup: Town(x)")

nairobiConnections = []
for town, roadDistance in getConnectedTowns("Nairobi"):
    nairobiConnections.append(f"{town} ({roadDistance} km)")
printAnswer("Which towns are connected to Nairobi?", ", ".join(nairobiConnections), "fact lookup: Road(Nairobi, y, d)")

printAnswer("What is the distance between Nairobi and Nakuru?", f"{getRoadDistance('Nairobi', 'Nakuru')} km",
            "fact lookup: Road(Nairobi, Nakuru, d)")

printAnswer("Is the road from Nakuru to Mau Summit open?", describeRoadStatus("Nakuru", "Mau Summit"),
            "inference: RoadCondition + condition rule => RoadOpen")

printAnswer("Where is the trailer?", getCurrentLocation(), "fact lookup: At(Trailer, x)")
printAnswer("What is the destination?", getDestination(), "fact lookup: Destination(Trailer, x)")

directMoves = []
for town, roadDistance in getUsableNeighbours(getCurrentLocation()):
    directMoves.append(f"{town} ({roadDistance} km)")
printAnswer("Which towns can the trailer drive to directly from where it is?", ", ".join(directMoves),
            "inference: CanUse(At, y)")

printAnswer("Which towns are reachable from the current location?", ", ".join(sorted(findReachableTowns(getCurrentLocation()))),
            "inference: Reachable rules, forward chaining")

printAnswer("Can the trailer reach Kisumu?", "Yes" if canReachDestination() else "No",
            "inference: Reachable(At, Destination)")

unavailableRoads = listUnavailableRoads()
printAnswer("Which roads are unavailable?", unavailableRoads if len(unavailableRoads) > 0 else "None: every known road is usable.",
            "inference: RoadClosed rule over all Road facts")

Q: Which towns exist?
A: Bomet, Kericho, Kisii, Kisumu, Mau Summit, Nairobi, Nakuru, Narok, Nyahururu, Nyeri
   [fact lookup: Town(x)]

Q: Which towns are connected to Nairobi?
A: Narok (141 km), Nyeri (151 km), Nakuru (161 km)
   [fact lookup: Road(Nairobi, y, d)]

Q: What is the distance between Nairobi and Nakuru?
A: 161 km
   [fact lookup: Road(Nairobi, Nakuru, d)]

Q: Is the road from Nakuru to Mau Summit open?
A: Open. Usable: the road is open to the trailer.
   [inference: RoadCondition + condition rule => RoadOpen]

Q: Where is the trailer?
A: Nairobi
   [fact lookup: At(Trailer, x)]

Q: What is the destination?
A: Kisumu
   [fact lookup: Destination(Trailer, x)]

Q: Which towns can the trailer drive to directly from where it is?
A: Narok (141 km), Nyeri (151 km), Nakuru (161 km)
   [inference: CanUse(At, y)]

Q: Which towns are reachable from the current location?
A: Bomet, Kericho, Kisii, Kisumu, Mau Summit, Nakuru, Narok, Nyahururu, Nyeri
   [inference: Reachable rules, forw

## Step 8: Querying the Knowledge Base to Make Decisions

The queries in Step 7 answered questions. Here the agent uses the KB to **decide** which route to drive. This is where the Uniform Cost Search from Practical Exercise 3 joins in.

**How the KB and UCS work together:**

```
 KB facts (Town, Road, RoadCondition)
        │  CanUse rule filters the roads
        ▼
 buildAvailableRoadGraph()  ──►  graph of usable roads only
        │
 KB facts (At, Destination) ──►  start and goal
        │
        ▼
 uniformCostSearch()  (Practical Exercise 3, reused)  ──►  cheapest route + cost + search tree
        │
        ▼
 planRoute() / explainPlan()  ──►  decision, explanation, next move
```

UCS never sees the raw road list. It only sees the graph the KB says is usable. When a fact changes, the graph changes, and so does the route UCS finds.

### 8.1 Build the road graph from the KB

`buildAvailableRoadGraph()` is our adaptation of Practical Exercise 3's `buildGraph()`. It produces the **same data structure**: `{town: [(neighbour, distance), ...]}` with neighbours sorted by distance and then name. That means the original UCS code can use it unchanged. The difference is where the data comes from. Practical Exercise 3 read the fixed `roadConnections` list. We ask the KB, and a directed arc is included only if the `CanUse` rule allows it. Every known town gets an entry, even one with no usable roads, so the search never looks up a missing key.

We also keep Practical Exercise 3's original `buildGraph()` so we can check that, while every road is open, the KB-built graph is identical to the Practical Exercise 3 graph.

In [18]:
def buildAvailableRoadGraph():
    """Adapted from Practical Exercise 3's buildGraph(): same structure, but built from usable roads in the KB."""
    availableRoadGraph = {}
    for town in sorted(knowledgeBase["Town"]):
        availableRoadGraph[town] = getUsableNeighbours(town)
    return availableRoadGraph


def buildGraph(roadConnections):
    """Practical Exercise 3's original graph construction, kept unchanged for comparison."""
    roadGraph = {}
    for firstTown, secondTown, roadDistance in roadConnections:
        if firstTown not in roadGraph:
            roadGraph[firstTown] = []
        if secondTown not in roadGraph:
            roadGraph[secondTown] = []
        # Practical Exercise 3 assumes this road can be travelled in both directions.
        roadGraph[firstTown].append((secondTown, roadDistance))
        roadGraph[secondTown].append((firstTown, roadDistance))

    for town in roadGraph:
        roadGraph[town].sort(key=lambda connection: (connection[1], connection[0]))

    return roadGraph


def countArcs(roadGraph):
    """Number of directed arcs in a graph."""
    arcCount = 0
    for town in roadGraph:
        arcCount = arcCount + len(roadGraph[town])
    return arcCount


availableRoadGraph = buildAvailableRoadGraph()
practicalThreeGraph = buildGraph(roadConnections)

print("Road graph built from the KB (usable roads only):")
for town, neighbours in availableRoadGraph.items():
    neighbourDescriptions = []
    for neighbourTown, roadDistance in neighbours:
        neighbourDescriptions.append(f"{neighbourTown} ({roadDistance})")
    print(f"  {town:<11} -> {', '.join(neighbourDescriptions)}")

print(f"\nDirected arcs in the KB graph: {countArcs(availableRoadGraph)}")
print("Identical to the Practical Exercise 3 graph while every road is open?", availableRoadGraph == practicalThreeGraph)
assert availableRoadGraph == practicalThreeGraph

Road graph built from the KB (usable roads only):
  Bomet       -> Kericho (77), Narok (81), Kisii (83)
  Kericho     -> Mau Summit (50), Bomet (77), Kisumu (84)
  Kisii       -> Bomet (83), Kisumu (114)
  Kisumu      -> Kericho (84), Kisii (114)
  Mau Summit  -> Kericho (50), Nakuru (62)
  Nairobi     -> Narok (141), Nyeri (151), Nakuru (161)
  Nakuru      -> Mau Summit (62), Nyahururu (64), Nairobi (161)
  Narok       -> Bomet (81), Nairobi (141)
  Nyahururu   -> Nakuru (64), Nyeri (104)
  Nyeri       -> Nyahururu (104), Nairobi (151)

Directed arcs in the KB graph: 24
Identical to the Practical Exercise 3 graph while every road is open? True


### 8.2 Helper functions reused from Practical Exercise 3

These four helpers are taken from Practical Exercise 3 unchanged: printing headings, showing the live frontier, rebuilding a route from the `predecessors` links, and looking up the cost of one road segment in a graph.

In [19]:
def printBanner(title, separatorCharacter="="):
    """Print a heading that makes the console output easier to scan."""
    print("\n" + separatorCharacter * 78)
    print(title)
    print(separatorCharacter * 78)


def formatFrontier(frontier, bestCosts, settledTowns):
    """Show the current best entries waiting in the priority queue."""
    liveFrontier = []

    for pathCost, insertionOrder, town in frontier:
        # A town may still have an old queue entry after a cheaper route was found.
        # Only show entries that match the best cost we currently know for that town.
        if town not in settledTowns:
            if pathCost == bestCosts[town]:
                liveFrontier.append((pathCost, insertionOrder, town))

    # This order matches the way the queue chooses its next entry.
    liveFrontier.sort()

    if len(liveFrontier) == 0:
        return "(empty)"

    frontierDescriptions = []
    for pathCost, insertionOrder, town in liveFrontier:
        frontierDescriptions.append(f"{town} [cost= {pathCost} km]")

    return ", ".join(frontierDescriptions)


def reconstructPath(predecessors, destinationTown):
    """Follow the saved previous-town links to rebuild the route."""
    route = []
    currentTown = destinationTown

    while currentTown is not None:
        route.append(currentTown)
        currentTown = predecessors[currentTown]
    # We followed the route backwards, so reverse it to get start -> destination.
    route.reverse()
    return route


def getEdgeCost(roadGraph, startingTown, destinationTown):
    """Return the distance for a road between two neighbouring towns."""
    for neighbourTown, roadDistance in roadGraph[startingTown]:
        if neighbourTown == destinationTown:
            return roadDistance

    raise ValueError(f"No road connects {startingTown} and {destinationTown} in this graph.")

### 8.3 Uniform Cost Search (reused from Practical Exercise 3)

The search logic is exactly the Practical Exercise 3 algorithm:
- the frontier is a priority queue of `(cost so far, tie-break number, town)`;
- `bestCosts` holds the cheapest known cost to each town, and `predecessors` holds the town we came from on that route;
- stale queue entries (a cheaper route was found later) and already-settled towns are skipped;
- the goal test happens when a town is **removed** from the frontier, not when it is first seen, which is what guarantees the cheapest route;
- `searchTreeRecord` records every road option considered and what was decided about it.

The only change is the `showTrace` parameter. Practical Exercise 3 always printed the full step-by-step trace. The agent replans several times in this notebook, so we print the full trace for the first plan and switch it off afterwards. Every `print` from the original goes through the small `trace()` function, which prints only when `showTrace` is `True`.

In [20]:
def uniformCostSearch(roadGraph, startTown, destinationTown, showTrace=True):
    """Practical Exercise 3's Uniform Cost Search, with an option to hide the step-by-step trace."""

    def trace(message=""):
        """Print one line of the search trace, but only if showTrace is True."""
        if showTrace:
            print(message)

    # The queue stores (total distance so far, tie-break number, town).
    tieBreaker = count()
    frontier = [(0, next(tieBreaker), startTown)]

    # bestCosts remembers the cheapest route found to each town.
    bestCosts = {startTown: 0}

    # predecessors remembers the town we came from on that cheapest route.
    predecessors = {startTown: None}

    # Once a town is settled, its cheapest distance is final for this graph.
    settledTowns = set()

    # A record of each road option considered, for printing the search tree later.
    searchTreeRecord = []
    selectedTownCount = 0
    expandedTownCount = 0

    if showTrace:
        printBanner("UNIFORM COST SEARCH")
    trace(f"Start: {startTown}")
    trace(f"Goal:  {destinationTown}")
    trace("Starting cost is 0 km because the trailer has not travelled yet.")
    trace(f"Frontier: {formatFrontier(frontier, bestCosts, settledTowns)}")
    trace("Settled towns: (none yet)")
    trace("\nImportant rule: we check for the goal when it is removed from the frontier.")
    trace("Finding the goal as a neighbour is not enough to stop, because a cheaper")
    trace("route to it might still be waiting to be found.")

    while len(frontier) > 0:
        # Remove whichever entry has the smallest total distance.
        pathCost, insertionOrder, currentTown = heapq.heappop(frontier)

        # If this town was already settled, this queue entry can be ignored.
        if currentTown in settledTowns:
            trace(f"\n[SKIP] {currentTown} at {pathCost} km was already settled.")
            continue

        # A better route may have been found after this older entry was added.
        if pathCost != bestCosts[currentTown]:
            trace(f"\n[SKIP] Old entry for {currentTown}: {pathCost} km.")
            trace(f"       The best known cost is now {bestCosts[currentTown]} km, so this entry is out of date.")
            continue

        selectedTownCount += 1
        if showTrace:
            printBanner(f"STEP {selectedTownCount}: TAKE {currentTown.upper()} FROM THE FRONTIER", "-")
        trace(f"We selected {currentTown} because its current cost is the lowest: {pathCost} km.")
        trace(f"Route so far: {' -> '.join(reconstructPath(predecessors, currentTown))}")
        trace(f"Other options waiting: {formatFrontier(frontier, bestCosts, settledTowns)}")

        # No cheaper route to this town can appear later, so we settle it now.
        settledTowns.add(currentTown)

        # Goal test happens only when a town is taken from the priority queue.
        if currentTown == destinationTown:
            trace(f"\nGOAL TEST: Is {currentTown} the destination {destinationTown}? YES.")
            trace(f"We reached the goal with a total path cost of {pathCost} km.")
            trace("Because this was the lowest-cost entry in the frontier, the route is optimal for this graph.")

            finalPath = reconstructPath(predecessors, destinationTown)
            return finalPath, pathCost, searchTreeRecord, selectedTownCount, expandedTownCount, settledTowns

        trace(f"\nGOAL TEST: Is {currentTown} the destination {destinationTown}? No.")
        trace(f"We will now check the roads leaving {currentTown}.")
        expandedTownCount += 1

        for neighbourTown, roadDistance in roadGraph[currentTown]:
            newPathCost = pathCost + roadDistance

            trace(f"\n  Road option: {currentTown} -> {neighbourTown} ({roadDistance} km)")
            trace(f"  Cost if we take this road: {pathCost} + {roadDistance} = {newPathCost} km.")

            if neighbourTown in settledTowns:
                trace(f"  Decision: ignore it. We've already settled {neighbourTown} at its cheapest cost.")
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "already settled; ignored"))

            elif neighbourTown not in bestCosts:
                # First route found to this town: remember it and add it to the frontier.
                bestCosts[neighbourTown] = newPathCost
                predecessors[neighbourTown] = currentTown
                heapq.heappush(frontier, (newPathCost, next(tieBreaker), neighbourTown))
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "new route"))
                trace(f"  Decision: add {neighbourTown} to the frontier at {newPathCost} km. This is our first route to it.")

            elif newPathCost < bestCosts[neighbourTown]:
                # Cheaper route found: update the cost and the previous town.
                previousBestCost = bestCosts[neighbourTown]
                bestCosts[neighbourTown] = newPathCost
                predecessors[neighbourTown] = currentTown
                heapq.heappush(frontier, (newPathCost, next(tieBreaker), neighbourTown))
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, f"improved from {previousBestCost} km"))
                trace(f"  Decision: keep this route. {newPathCost} km is cheaper than the previous best of {previousBestCost} km.")
                trace("  The old queue entry can stay there for now; we'll skip it if it comes up later.")

            elif newPathCost == bestCosts[neighbourTown]:
                # A tie does not improve the best cost, so no new queue entry is needed.
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "same cost; no update"))
                trace(f"  Decision: the cost ties the current best of {bestCosts[neighbourTown]} km, so no update is needed.")

            else:
                # This option costs more than the route we already know.
                searchTreeRecord.append((currentTown, neighbourTown, newPathCost, "more expensive; discarded"))
                trace(f"  Decision: discard it. We already know a cheaper route costing {bestCosts[neighbourTown]} km.")

        trace(f"\nFrontier after checking {currentTown}: {formatFrontier(frontier, bestCosts, settledTowns)}")
        trace(f"Towns settled so far: {', '.join(sorted(settledTowns))}")

    # The frontier emptied without settling the goal, so no route exists in this graph.
    trace(f"No route from {startTown} to {destinationTown} was found in this graph.")
    return None, float("inf"), searchTreeRecord, selectedTownCount, expandedTownCount, settledTowns

### 8.4 The agent's decision procedure

`planRoute()` is where the agent decides what to do. It follows these steps, all based on the KB:

1. **Ask** the KB where the trailer is (`At`) and where it must go (`Destination`).
2. If either is unknown, it cannot plan. If they are the same town, no movement is needed.
3. **Infer** with the reachability rules whether the destination can be reached at all. If not, it reports which roads are unavailable instead of searching.
4. **Build** the graph of usable roads from the KB.
5. **Search** that graph with UCS (the lowest-cost rule) and record the chosen route, its cost and the reason.

`explainPlan()` then presents the decision. It shows the route as a segment table (the same layout as Practical Exercise 3's final answer), the reason, and the trailer's **next move**.

In [21]:
def planRoute(showTrace=False):
    """The agent's decision procedure: ask the KB, check reachability, build the usable graph, run UCS."""
    currentLocation = getCurrentLocation()
    destination = getDestination()

    plan = {
        "start": currentLocation,
        "destination": destination,
        "route": None,
        "cost": None,
        "roadGraph": None,
        "searchTreeRecord": [],
        "reason": "",
    }

    if currentLocation is None or destination is None:
        plan["reason"] = "The KB does not say where the trailer is or where it must go, so no plan can be made."
        return plan

    if currentLocation == destination:
        plan["route"] = [currentLocation]
        plan["cost"] = 0
        plan["reason"] = f"At({trailer}, {currentLocation}) and Destination({trailer}, {destination}) agree: no movement is needed."
        return plan

    if not canReachDestination():
        plan["reason"] = (f"The reachability rules show no chain of usable roads from {currentLocation} to {destination}. "
                          f"Unavailable roads: {listUnavailableRoads()}")
        return plan

    roadGraph = buildAvailableRoadGraph()
    route, cost, searchTreeRecord, selectedTownCount, expandedTownCount, settledTowns = uniformCostSearch(
        roadGraph, currentLocation, destination, showTrace
    )

    usableRoadCount = countArcs(roadGraph) // 2
    totalRoadCount = len(knowledgeBase["Road"]) // 2
    plan["route"] = route
    plan["cost"] = cost
    plan["roadGraph"] = roadGraph
    plan["searchTreeRecord"] = searchTreeRecord
    plan["reason"] = (f"Using only the {usableRoadCount} of {totalRoadCount} roads the KB marks as usable, "
                      f"UCS settled {selectedTownCount} towns and found this to be the lowest-cost route.")
    return plan

In [22]:
def explainPlan(plan):
    """Present the agent's decision: route table (Practical Exercise 3 layout), reason and next move."""
    printBanner(f"AGENT DECISION: {plan['start']} -> {plan['destination']}")

    if plan["route"] is None:
        print("Decision: do not move. No usable route exists.")
        print("Reason:", plan["reason"])
        return

    route = plan["route"]
    print("Route:", " -> ".join(route))

    if len(route) > 1:
        print()
        print(f"{'Road segment':<34}{'Distance (km)':>16}{'Cumulative (km)':>20}")
        print("-" * 70)
        cumulativeDistance = 0
        distanceParts = []
        for pathIndex in range(len(route) - 1):
            startingTown = route[pathIndex]
            nextTown = route[pathIndex + 1]
            roadDistance = getEdgeCost(plan["roadGraph"], startingTown, nextTown)
            cumulativeDistance = cumulativeDistance + roadDistance
            distanceParts.append(str(roadDistance))
            roadSegment = startingTown + " -> " + nextTown
            print(f"{roadSegment:<34}{roadDistance:>16}{cumulativeDistance:>20}")
        print()
        print("Total distance:", " + ".join(distanceParts), "=", cumulativeDistance, "km")
        print("Matches the cost returned by UCS:", cumulativeDistance == plan["cost"])
        print(f"Next move: drive from {route[0]} to {route[1]}.")

    print("Reason:", plan["reason"])

### 8.5 The agent plans its route from Nairobi to Kisumu

The first plan prints the full Practical Exercise 3 search trace, so we can watch UCS work on the graph that came from the KB.

In [23]:
baselinePlan = planRoute(showTrace=True)


UNIFORM COST SEARCH
Start: Nairobi
Goal:  Kisumu
Starting cost is 0 km because the trailer has not travelled yet.
Frontier: Nairobi [cost= 0 km]
Settled towns: (none yet)

Important rule: we check for the goal when it is removed from the frontier.
Finding the goal as a neighbour is not enough to stop, because a cheaper
route to it might still be waiting to be found.

------------------------------------------------------------------------------
STEP 1: TAKE NAIROBI FROM THE FRONTIER
------------------------------------------------------------------------------
We selected Nairobi because its current cost is the lowest: 0 km.
Route so far: Nairobi
Other options waiting: (empty)

GOAL TEST: Is Nairobi the destination Kisumu? No.
We will now check the roads leaving Nairobi.

  Road option: Nairobi -> Narok (141 km)
  Cost if we take this road: 0 + 141 = 141 km.
  Decision: add Narok to the frontier at 141 km. This is our first route to it.

  Road option: Nairobi -> Nyeri (151 km)
  Cost

In [24]:
explainPlan(baselinePlan)

assert baselinePlan["route"] == ["Nairobi", "Nakuru", "Mau Summit", "Kericho", "Kisumu"]
assert baselinePlan["cost"] == 357
print("\nCheck passed: the agent chose Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu (357 km), as in Practical Exercise 3.")


AGENT DECISION: Nairobi -> Kisumu
Route: Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu

Road segment                         Distance (km)     Cumulative (km)
----------------------------------------------------------------------
Nairobi -> Nakuru                              161                 161
Nakuru -> Mau Summit                            62                 223
Mau Summit -> Kericho                           50                 273
Kericho -> Kisumu                               84                 357

Total distance: 161 + 62 + 50 + 84 = 357 km
Matches the cost returned by UCS: True
Next move: drive from Nairobi to Nakuru.
Reason: Using only the 12 of 12 roads the KB marks as usable, UCS settled 10 towns and found this to be the lowest-cost route.

Check passed: the agent chose Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu (357 km), as in Practical Exercise 3.


### 8.6 Verifying the decision (from Practical Exercise 3)

As in Practical Exercise 3, we list every **simple route** (no town visited twice) through the graph the agent used and check that none is cheaper than the UCS answer. We also print the search tree record that UCS kept.

In [25]:
def findAllSimplePaths(roadGraph, currentTown, destinationTown, currentPath, currentPathCost, foundRoutes):
    """Collect every route to the destination without visiting a town twice (Practical Exercise 3)."""
    currentPath.append(currentTown)

    if currentTown == destinationTown:
        # Save a copy, because currentPath changes as the recursion unwinds.
        foundRoutes.append((currentPath.copy(), currentPathCost))
    else:
        for neighbourTown, roadDistance in roadGraph[currentTown]:
            if neighbourTown not in currentPath:
                findAllSimplePaths(roadGraph, neighbourTown, destinationTown,
                                   currentPath, currentPathCost + roadDistance, foundRoutes)

    # Remove this town before returning so other routes can use it.
    currentPath.pop()


def listAllSimpleRoutes(plan):
    """Print every simple route in the plan's graph, cheapest first, and return them."""
    allSimpleRoutes = []
    findAllSimplePaths(plan["roadGraph"], plan["start"], plan["destination"], [], 0, allSimpleRoutes)
    allSimpleRoutes.sort(key=lambda routeDetails: routeDetails[1])

    for routePath, routeCost in allSimpleRoutes:
        marker = "  <-- chosen by the agent" if routePath == plan["route"] else ""
        print(f"{routeCost:>4} km  {' -> '.join(routePath)}{marker}")
    return allSimpleRoutes


printBanner("VERIFICATION: ALL SIMPLE ROUTES IN THE AGENT'S GRAPH", "-")
baselineRoutes = listAllSimpleRoutes(baselinePlan)
assert baselineRoutes[0][1] == baselinePlan["cost"], "A cheaper route exists than the one UCS found."
print(f"\nCheapest simple route: {baselineRoutes[0][1]} km | UCS result: {baselinePlan['cost']} km | agree: True")

print("\nSearch tree record (road option -> cumulative cost):")
for previousTown, nextTown, newPathCost, decisionNote in baselinePlan["searchTreeRecord"]:
    print(f"  {previousTown:<11} -> {nextTown:<11} cost = {newPathCost:<4} km | {decisionNote}")


------------------------------------------------------------------------------
VERIFICATION: ALL SIMPLE ROUTES IN THE AGENT'S GRAPH
------------------------------------------------------------------------------
 357 km  Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu  <-- chosen by the agent
 383 km  Nairobi -> Narok -> Bomet -> Kericho -> Kisumu
 419 km  Nairobi -> Narok -> Bomet -> Kisii -> Kisumu
 515 km  Nairobi -> Nyeri -> Nyahururu -> Nakuru -> Mau Summit -> Kericho -> Kisumu
 547 km  Nairobi -> Nakuru -> Mau Summit -> Kericho -> Bomet -> Kisii -> Kisumu
 705 km  Nairobi -> Nyeri -> Nyahururu -> Nakuru -> Mau Summit -> Kericho -> Bomet -> Kisii -> Kisumu

Cheapest simple route: 357 km | UCS result: 357 km | agree: True

Search tree record (road option -> cumulative cost):
  Nairobi     -> Narok       cost = 141  km | new route
  Nairobi     -> Nyeri       cost = 151  km | new route
  Nairobi     -> Nakuru      cost = 161  km | new route
  Narok       -> Bomet       cost = 2

## Step 9: Debugging the Knowledge Base

Following the lecturer's approach, we test the agent with a list of cases, including faulty or unexpected ones, and check that its answers are sensible. We also change the world and check that the agent's decisions change with it.

### 9.1 Querying known and unknown towns

The list mixes correct names, names with different spacing or case, and towns that are not in the KB.

In [26]:
testTowns = ["Nairobi", "kisumu", "  mau summit ", "Mombasa", "Eldoret", ""]

print("--- Debugging town queries ---")
for town in testTowns:
    shownName = repr(town)
    if isKnownTown(town):
        connections = []
        for neighbourTown, roadDistance in getConnectedTowns(town):
            connections.append(f"{neighbourTown} ({roadDistance} km)")
        print(f"{shownName:<16} -> known as {normalizeName(town)}; connected to {', '.join(connections)}")
    else:
        print(f"{shownName:<16} -> Unknown town: not in the knowledge base.")

--- Debugging town queries ---
'Nairobi'        -> known as Nairobi; connected to Narok (141 km), Nyeri (151 km), Nakuru (161 km)
'kisumu'         -> known as Kisumu; connected to Kericho (84 km), Kisii (114 km)
'  mau summit '  -> known as Mau Summit; connected to Kericho (50 km), Nakuru (62 km)
'Mombasa'        -> Unknown town: not in the knowledge base.
'Eldoret'        -> Unknown town: not in the knowledge base.
''               -> Unknown town: not in the knowledge base.


### 9.2 Unknown roads and invalid updates

We ask about road pairs the KB does not contain. We also try to tell the KB things that would make it inconsistent: setting the condition of a road that does not exist, and setting a destination that is not a known town. Both should be rejected, and the KB should be left unchanged.

In [27]:
testRoads = [("Nairobi", "Nakuru"), ("Nakuru", "Nairobi"), ("Nairobi", "Kisumu"), ("Nyeri", "Kisii"), ("Nairobi", "Mombasa")]

print("--- Debugging road queries ---")
for fromTown, toTown in testRoads:
    roadDistance = getRoadDistance(fromTown, toTown)
    distanceText = f"{roadDistance} km" if roadDistance is not None else "no Road fact"
    print(f"{fromTown + ' -> ' + toTown:<20} distance: {distanceText:<13} status: {describeRoadStatus(fromTown, toTown)}")

print("\n--- Debugging invalid updates ---")
roadFactsBefore = dict(knowledgeBase["RoadCondition"])
print("Try: close the (non-existent) Nairobi-Kisumu road")
setRoadCondition("Nairobi", "Kisumu", "Closed")
print("Try: change the destination to Mombasa")
setDestination("Mombasa")

assert knowledgeBase["RoadCondition"] == roadFactsBefore
assert getDestination() == "Kisumu"
print("\nCheck passed: invalid updates were rejected and the KB is unchanged.")

--- Debugging road queries ---
Nairobi -> Nakuru    distance: 161 km        status: Open. Usable: the road is open to the trailer.
Nakuru -> Nairobi    distance: 161 km        status: Open. Usable: the road is open to the trailer.
Nairobi -> Kisumu    distance: no Road fact  status: Unknown road: the KB has no Road fact from Nairobi to Kisumu.
Nyeri -> Kisii       distance: no Road fact  status: Unknown road: the KB has no Road fact from Nyeri to Kisii.
Nairobi -> Mombasa   distance: no Road fact  status: Unknown road: the KB has no Road fact from Nairobi to Mombasa.

--- Debugging invalid updates ---
Try: close the (non-existent) Nairobi-Kisumu road
  Rejected: the KB has no road between Nairobi and Kisumu.
Try: change the destination to Mombasa
  Rejected: 'Mombasa' is not a known town, so Destination(Trailer, Mombasa) was not added.

Check passed: invalid updates were rejected and the KB is unchanged.


### 9.3 Testing the road condition rules

This is the direct equivalent of the lecturer's `test_cases` loop over traffic light colours. Known conditions should follow their rule. Conditions with odd capitalisation should be normalised. Unknown conditions (`Landslide`, `Foggy`) should fall back to the cautious default.

In [28]:
testConditions = ["Open", "wet", "Under Repair", "Closed", "Flooded", "weight restricted", "Landslide", "Foggy"]

print("--- Debugging road condition rules ---")
for condition in testConditions:
    rule = getConditionRule(condition)
    print(f"Condition: {condition:<18} -> usable: {str(rule['usable']):<5} | {rule['advice']}")

--- Debugging road condition rules ---
Condition: Open               -> usable: True  | Usable: the road is open to the trailer.
Condition: wet                -> usable: True  | Usable: reduce speed, the surface is slippery.
Condition: Under Repair       -> usable: True  | Usable: expect delays from roadworks.
Condition: Closed             -> usable: False | Not usable: the road is closed to all traffic.
Condition: Flooded            -> usable: False | Not usable: the trailer cannot safely cross flood water.
Condition: weight restricted  -> usable: False | Not usable: the loaded trailer exceeds the weight limit.
Condition: Landslide          -> usable: False | Unknown condition: treat the road as unusable until confirmed safe.
Condition: Foggy              -> usable: False | Unknown condition: treat the road as unusable until confirmed safe.


### 9.4 Main scenario: a road closure forces the agent to replan

The trailer is in Nairobi and has planned the 357 km route through Nakuru and Mau Summit. Then a **new observation** arrives: *the road between Nakuru and Mau Summit is closed.*

`observeRoadCondition()` is how the agent takes in an observation (a *percept*). It tells the KB the new fact and reports what the rules now conclude. The agent then:

1. updates its KB;
2. recognises that the road is unavailable;
3. builds the new graph of usable roads;
4. replans with UCS;
5. finds the alternative route;
6. explains why the decision changed.

In [29]:
def observeRoadCondition(firstTown, secondTown, condition):
    """A percept: the trailer observes a road's condition, tells the KB, and reports the rule's conclusion."""
    firstTown = normalizeName(firstTown)
    secondTown = normalizeName(secondTown)
    condition = normalizeName(condition)
    print(f"Observation: the road between {firstTown} and {secondTown} is {condition}.")

    if not setRoadCondition(firstTown, secondTown, condition):
        return False

    print(f"  KB updated: RoadCondition({firstTown}, {secondTown}, {condition}) and RoadCondition({secondTown}, {firstTown}, {condition})")
    print(f"  Rule applied: {getConditionRule(condition)['advice']}")
    print(f"  Inferred: CanUse({firstTown}, {secondTown}) = {canUseRoad(firstTown, secondTown)}")
    return True


def explainRouteChange(previousPlan, newPlan):
    """Explain why the new plan differs from the previous one, using the current KB."""
    printBanner("WHY THE DECISION CHANGED", "-")
    previousRoute = previousPlan["route"]

    blockedSegments = []
    for pathIndex in range(len(previousRoute) - 1):
        fromTown = previousRoute[pathIndex]
        toTown = previousRoute[pathIndex + 1]
        if not canUseRoad(fromTown, toTown):
            condition = knowledgeBase["RoadCondition"][(fromTown, toTown)]
            blockedSegments.append(f"{fromTown} -> {toTown} is now {condition}")

    print("Previous route:", " -> ".join(previousRoute), f"({previousPlan['cost']} km)")
    if len(blockedSegments) == 0:
        print("Every road on the previous route is still usable.")
    else:
        print("The previous route can no longer be used because:", "; ".join(blockedSegments))

    if newPlan["route"] is None:
        print("No usable alternative exists, so the trailer must wait for a road to reopen.")
    else:
        print("New route:     ", " -> ".join(newPlan["route"]), f"({newPlan['cost']} km)")
        print(f"Change in distance: {newPlan['cost'] - previousPlan['cost']:+} km")
        print("Decision: follow the new route. It is the cheapest route left using only roads the KB says are usable.")

**Before the observation**, the agent's plan is the 357 km route (from Step 8).

In [30]:
planBeforeClosure = planRoute()
print("Plan before closure:", " -> ".join(planBeforeClosure["route"]), f"({planBeforeClosure['cost']} km)")

Plan before closure: Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu (357 km)


**Steps 1 and 2: update the KB and recognise that the road is unavailable.**

In [31]:
observeRoadCondition("Nakuru", "Mau Summit", "Closed")

print()
printAnswer("Is the road from Nakuru to Mau Summit open?", describeRoadStatus("Nakuru", "Mau Summit"), "inference")
printAnswer("Is the road from Mau Summit to Nakuru open?", describeRoadStatus("Mau Summit", "Nakuru"), "inference")
printAnswer("Which roads are unavailable?", listUnavailableRoads(), "inference: RoadClosed rule")
printAnswer("Can the trailer still reach Kisumu?", "Yes" if canReachDestination() else "No", "inference: Reachable rules")

Observation: the road between Nakuru and Mau Summit is Closed.
  KB updated: RoadCondition(Nakuru, Mau Summit, Closed) and RoadCondition(Mau Summit, Nakuru, Closed)
  Rule applied: Not usable: the road is closed to all traffic.
  Inferred: CanUse(Nakuru, Mau Summit) = False

Q: Is the road from Nakuru to Mau Summit open?
A: Closed. Not usable: the road is closed to all traffic.
   [inference]

Q: Is the road from Mau Summit to Nakuru open?
A: Closed. Not usable: the road is closed to all traffic.
   [inference]

Q: Which roads are unavailable?
A: [('Mau Summit', 'Nakuru', 'Closed')]
   [inference: RoadClosed rule]

Q: Can the trailer still reach Kisumu?
A: Yes
   [inference: Reachable rules]



**Step 3: obtain the updated network of usable roads.** The two directed arcs for Nakuru ↔ Mau Summit should have disappeared from the graph that UCS will search.

In [32]:
graphAfterClosure = buildAvailableRoadGraph()

print("Nakuru's usable neighbours:    ", graphAfterClosure["Nakuru"])
print("Mau Summit's usable neighbours:", graphAfterClosure["Mau Summit"])
print(f"Directed arcs in the usable graph: {countArcs(graphAfterClosure)} (was {countArcs(practicalThreeGraph)})")

assert ("Mau Summit", 62) not in graphAfterClosure["Nakuru"]
assert ("Nakuru", 62) not in graphAfterClosure["Mau Summit"]
assert countArcs(graphAfterClosure) == 22

Nakuru's usable neighbours:     [('Nyahururu', 64), ('Nairobi', 161)]
Mau Summit's usable neighbours: [('Kericho', 50)]
Directed arcs in the usable graph: 22 (was 24)


**Steps 4–6: replan with UCS, identify the alternative route and explain the decision.**

In [33]:
planAfterClosure = planRoute()
explainPlan(planAfterClosure)
explainRouteChange(planBeforeClosure, planAfterClosure)

assert planAfterClosure["route"] == ["Nairobi", "Narok", "Bomet", "Kericho", "Kisumu"]
assert planAfterClosure["cost"] == 383
print("\nCheck passed: after the closure the agent chose Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (383 km).")


AGENT DECISION: Nairobi -> Kisumu
Route: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu

Road segment                         Distance (km)     Cumulative (km)
----------------------------------------------------------------------
Nairobi -> Narok                               141                 141
Narok -> Bomet                                  81                 222
Bomet -> Kericho                                77                 299
Kericho -> Kisumu                               84                 383

Total distance: 141 + 81 + 77 + 84 = 383 km
Matches the cost returned by UCS: True
Next move: drive from Nairobi to Narok.
Reason: Using only the 11 of 12 roads the KB marks as usable, UCS settled 10 towns and found this to be the lowest-cost route.

------------------------------------------------------------------------------
WHY THE DECISION CHANGED
------------------------------------------------------------------------------
Previous route: Nairobi -> Nakuru -> Mau Summit -

In [34]:
printBanner("VERIFICATION: ALL SIMPLE ROUTES AFTER THE CLOSURE", "-")
routesAfterClosure = listAllSimpleRoutes(planAfterClosure)
assert routesAfterClosure[0][1] == 383


------------------------------------------------------------------------------
VERIFICATION: ALL SIMPLE ROUTES AFTER THE CLOSURE
------------------------------------------------------------------------------
 383 km  Nairobi -> Narok -> Bomet -> Kericho -> Kisumu  <-- chosen by the agent
 419 km  Nairobi -> Narok -> Bomet -> Kisii -> Kisumu


**Why the KB must drive the planner.** For comparison, we run the same UCS on Practical Exercise 3's original, unchanged graph, which knows nothing about the closure. It still recommends the closed road. This is exactly the mistake the knowledge-based agent avoids.

In [35]:
staticRoute, staticCost, staticRecord, staticSelected, staticExpanded, staticSettled = uniformCostSearch(
    practicalThreeGraph, "Nairobi", "Kisumu", showTrace=False
)
print("UCS on the original Practical Exercise 3 graph (ignores the KB):", " -> ".join(staticRoute), f"({staticCost} km)")
print("Does that route use the closed Nakuru -> Mau Summit road?", "Mau Summit" in staticRoute and "Nakuru" in staticRoute)
print("UCS on the KB's usable graph:                         ", " -> ".join(planAfterClosure["route"]), f"({planAfterClosure['cost']} km)")

assert staticCost == 357
assert planAfterClosure["cost"] != staticCost

UCS on the original Practical Exercise 3 graph (ignores the KB): Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu (357 km)
Does that route use the closed Nakuru -> Mau Summit road? True
UCS on the KB's usable graph:                          Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (383 km)


**Reopening the road.** A later observation says the road is open again. Nothing else needs to change: the rules recompute `CanUse`, the graph regains its two arcs, and UCS returns to the 357 km route.

In [36]:
observeRoadCondition("Nakuru", "Mau Summit", "Open")

planAfterReopening = planRoute()
print()
print("Unavailable roads:", listUnavailableRoads() if len(listUnavailableRoads()) > 0 else "none")
print("Usable graph identical to Practical Exercise 3 again?", buildAvailableRoadGraph() == practicalThreeGraph)
explainPlan(planAfterReopening)

assert isRoadOpen("Nakuru", "Mau Summit") and isRoadOpen("Mau Summit", "Nakuru")
assert planAfterReopening["route"] == baselinePlan["route"]
assert planAfterReopening["cost"] == 357
print("\nCheck passed: reopening the road restored the original 357 km route.")

Observation: the road between Nakuru and Mau Summit is Open.
  KB updated: RoadCondition(Nakuru, Mau Summit, Open) and RoadCondition(Mau Summit, Nakuru, Open)
  Rule applied: Usable: the road is open to the trailer.
  Inferred: CanUse(Nakuru, Mau Summit) = True

Unavailable roads: none
Usable graph identical to Practical Exercise 3 again? True

AGENT DECISION: Nairobi -> Kisumu
Route: Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu

Road segment                         Distance (km)     Cumulative (km)
----------------------------------------------------------------------
Nairobi -> Nakuru                              161                 161
Nakuru -> Mau Summit                            62                 223
Mau Summit -> Kericho                           50                 273
Kericho -> Kisumu                               84                 357

Total distance: 161 + 62 + 50 + 84 = 357 km
Matches the cost returned by UCS: True
Next move: drive from Nairobi to Nakuru.
Reason:

### 9.5 An unreachable destination

Kisumu has only two roads: from Kericho and from Kisii. If floods cut both, the reachability rules should conclude that Kisumu cannot be reached. The agent should then decide **not to move**, rather than search pointlessly or report a route that does not exist. We then reopen both roads.

In [37]:
observeRoadCondition("Kericho", "Kisumu", "Flooded")
observeRoadCondition("Kisii", "Kisumu", "Flooded")

print()
printAnswer("Which towns are reachable from Nairobi?", ", ".join(sorted(findReachableTowns("Nairobi"))), "inference: Reachable rules")
printAnswer("Can the trailer reach Kisumu?", "Yes" if canReachDestination() else "No", "inference")

floodedPlan = planRoute()
explainPlan(floodedPlan)
assert floodedPlan["route"] is None
assert "Kisumu" not in findReachableTowns("Nairobi")

Observation: the road between Kericho and Kisumu is Flooded.
  KB updated: RoadCondition(Kericho, Kisumu, Flooded) and RoadCondition(Kisumu, Kericho, Flooded)
  Rule applied: Not usable: the trailer cannot safely cross flood water.
  Inferred: CanUse(Kericho, Kisumu) = False
Observation: the road between Kisii and Kisumu is Flooded.
  KB updated: RoadCondition(Kisii, Kisumu, Flooded) and RoadCondition(Kisumu, Kisii, Flooded)
  Rule applied: Not usable: the trailer cannot safely cross flood water.
  Inferred: CanUse(Kisii, Kisumu) = False

Q: Which towns are reachable from Nairobi?
A: Bomet, Kericho, Kisii, Mau Summit, Nakuru, Narok, Nyahururu, Nyeri
   [inference: Reachable rules]

Q: Can the trailer reach Kisumu?
A: No
   [inference]


AGENT DECISION: Nairobi -> Kisumu
Decision: do not move. No usable route exists.
Reason: The reachability rules show no chain of usable roads from Nairobi to Kisumu. Unavailable roads: [('Kericho', 'Kisumu', 'Flooded'), ('Kisii', 'Kisumu', 'Flooded')]


For comparison, Practical Exercise 3's UCS on this flooded graph would only find out that Kisumu is unreachable after emptying its whole frontier. The KB answers the same question with the reachability rules before any search is run. Now the floods recede:

In [38]:
observeRoadCondition("Kericho", "Kisumu", "Open")
observeRoadCondition("Kisii", "Kisumu", "Open")

assert canReachDestination()
assert planRoute()["cost"] == 357
print("\nCheck passed: Kisumu is reachable again and the 357 km route is restored.")

Observation: the road between Kericho and Kisumu is Open.
  KB updated: RoadCondition(Kericho, Kisumu, Open) and RoadCondition(Kisumu, Kericho, Open)
  Rule applied: Usable: the road is open to the trailer.
  Inferred: CanUse(Kericho, Kisumu) = True
Observation: the road between Kisii and Kisumu is Open.
  KB updated: RoadCondition(Kisii, Kisumu, Open) and RoadCondition(Kisumu, Kisii, Open)
  Rule applied: Usable: the road is open to the trailer.
  Inferred: CanUse(Kisii, Kisumu) = True

Check passed: Kisumu is reachable again and the 357 km route is restored.


### 9.6 A destination the trailer has already reached

If `At` and `Destination` name the same town, the correct decision is to stay put. We test this by moving the trailer to Kisumu in the KB and then putting it back.

In [39]:
setTrailerLocation("Kisumu")
explainPlan(planRoute())
assert planRoute()["cost"] == 0

setTrailerLocation("Nairobi")
print("\nTrailer location restored to:", getCurrentLocation())


AGENT DECISION: Kisumu -> Kisumu
Route: Kisumu
Reason: At(Trailer, Kisumu) and Destination(Trailer, Kisumu) agree: no movement is needed.

Trailer location restored to: Nairobi


### 9.7 Debugging summary

A final run of the key checks against the current KB. Every test above changed the KB and then restored it, so these checks also confirm that the KB is back in its original state.

In [40]:
finalChecks = [
    ("KB holds 10 towns", len(knowledgeBase["Town"]) == 10),
    ("KB holds 24 directed Road facts", len(knowledgeBase["Road"]) == 24),
    ("All roads usable again", len(listUnavailableRoads()) == 0),
    ("KB graph equals the Practical Exercise 3 graph", buildAvailableRoadGraph() == practicalThreeGraph),
    ("Trailer is At Nairobi", getCurrentLocation() == "Nairobi"),
    ("Destination is Kisumu", getDestination() == "Kisumu"),
    ("Baseline route is 357 km", planRoute()["cost"] == 357),
    ("Closure gave 383 km via Narok and Bomet", planAfterClosure["cost"] == 383),
    ("Flooding made Kisumu unreachable", floodedPlan["route"] is None),
    ("Unknown condition treated as unusable", getConditionRule("Landslide")["usable"] is False),
]

for checkDescription, checkPassed in finalChecks:
    print(f"{'PASS' if checkPassed else 'FAIL'}  {checkDescription}")

allChecksPassed = True
for checkDescription, checkPassed in finalChecks:
    if not checkPassed:
        allChecksPassed = False
assert allChecksPassed

PASS  KB holds 10 towns


PASS  KB holds 24 directed Road facts
PASS  All roads usable again
PASS  KB graph equals the Practical Exercise 3 graph
PASS  Trailer is At Nairobi
PASS  Destination is Kisumu
PASS  Baseline route is 357 km
PASS  Closure gave 383 km via Narok and Bomet
PASS  Flooding made Kisumu unreachable
PASS  Unknown condition treated as unusable


## Optional Extension: Driving the Route and Replanning on the Way

*This section goes beyond the core requirements.*

So far the trailer replanned while standing in Nairobi. A more realistic agent **acts** one step at a time. It drives one road, updates `At(Trailer, x)`, takes in any new observations at its new town, and then replans from where it now is. This is the classic knowledge-based agent loop: **perceive → tell the KB → ask the KB what to do → act**.

`driveToDestination()` takes a dictionary of observations the trailer will make on arriving at certain towns. Here the trailer only learns about the Nakuru–Mau Summit closure *after* it has already reached Nakuru.

In [41]:
def driveToDestination(observationsOnArrival, maximumMoves=20):
    """
    Drive one road at a time, replanning from the KB before every move.
    observationsOnArrival maps a town to a list of (firstTown, secondTown, condition) percepts
    the trailer makes when it arrives there.
    """
    journey = [getCurrentLocation()]
    distanceTravelled = 0

    for moveNumber in range(1, maximumMoves + 1):
        currentLocation = getCurrentLocation()
        if currentLocation == getDestination():
            print(f"\nArrived at {currentLocation}.")
            break

        # Ask: what does the KB say is the best route from here?
        plan = planRoute()
        if plan["route"] is None:
            print(f"\nStuck at {currentLocation}: {plan['reason']}")
            break

        # Act: drive only the first road of the plan.
        nextTown = plan["route"][1]
        roadDistance = getRoadDistance(currentLocation, nextTown)
        print(f"\nMove {moveNumber}: current plan {' -> '.join(plan['route'])} ({plan['cost']} km)")
        print(f"  Drive {currentLocation} -> {nextTown} ({roadDistance} km)")
        setTrailerLocation(nextTown)
        distanceTravelled = distanceTravelled + roadDistance
        journey.append(nextTown)

        # Perceive and tell: record anything observed on arrival.
        for firstTown, secondTown, condition in observationsOnArrival.get(nextTown, []):
            print("  ", end="")
            observeRoadCondition(firstTown, secondTown, condition)

    return journey, distanceTravelled


observationsOnArrival = {"Nakuru": [("Nakuru", "Mau Summit", "Closed")]}
journey, distanceTravelled = driveToDestination(observationsOnArrival)

print("\nJourney driven:", " -> ".join(journey))
print("Total distance driven:", distanceTravelled, "km")


Move 1: current plan Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu (357 km)
  Drive Nairobi -> Nakuru (161 km)
  Observation: the road between Nakuru and Mau Summit is Closed.
  KB updated: RoadCondition(Nakuru, Mau Summit, Closed) and RoadCondition(Mau Summit, Nakuru, Closed)
  Rule applied: Not usable: the road is closed to all traffic.
  Inferred: CanUse(Nakuru, Mau Summit) = False

Move 2: current plan Nakuru -> Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (544 km)
  Drive Nakuru -> Nairobi (161 km)

Move 3: current plan Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (383 km)
  Drive Nairobi -> Narok (141 km)

Move 4: current plan Narok -> Bomet -> Kericho -> Kisumu (242 km)
  Drive Narok -> Bomet (81 km)

Move 5: current plan Bomet -> Kericho -> Kisumu (161 km)
  Drive Bomet -> Kericho (77 km)

Move 6: current plan Kericho -> Kisumu (84 km)
  Drive Kericho -> Kisumu (84 km)

Arrived at Kisumu.

Journey driven: Nairobi -> Nakuru -> Nairobi -> Narok -> Bomet -> Kericho -> 

The trailer drove 161 km to Nakuru before it learned of the closure. From Nakuru the cheapest usable route turned out to be back through Nairobi and then via Narok, Bomet and Kericho. The whole journey came to 161 + 544 = 705 km, compared with the 383 km it would have driven had it known about the closure while still in Nairobi (both figures are checked in the next cell). This shows why **up-to-date knowledge is valuable**: the same rules give a much better decision when the facts reach the agent earlier.

We now restore the KB to the original problem instance.

In [42]:
assert journey == ["Nairobi", "Nakuru", "Nairobi", "Narok", "Bomet", "Kericho", "Kisumu"]
assert distanceTravelled == 161 + 544

setRoadCondition("Nakuru", "Mau Summit", "Open")
setTrailerLocation("Nairobi")
print("KB restored: At(Trailer,", getCurrentLocation() + "); unavailable roads:", listUnavailableRoads() or "none")
assert planRoute()["cost"] == 357

KB restored: At(Trailer, Nairobi); unavailable roads: none


## Step 10: Final Reflection

**How knowledge was represented.** Following the lecturer's approach, the knowledge base is a Python dictionary whose keys are predicate names (`Town`, `Road`, `RoadCondition`, `At`, `Destination`) and whose values hold the facts for each predicate. Constants (`Trailer`, the ten towns, the road conditions) are strings kept in one standard form by `normalizeName()`. Each two-way road from Practical Exercise 3 is stored as two directed `Road` facts, which mirrors the two arcs that Practical Exercise 3's `buildGraph()` created.

**How facts and rules were used.** *Facts* describe this particular world and change over time: road conditions, the trailer's location. *Rules* are general and never change: the `roadConditionRules` dictionary (the counterpart of the lecturer's traffic light `rules`, with the same cautious default for unknown inputs), and small rule functions for `CanUse`, `RoadOpen`, `RoadClosed` and `Reachable`. Derived predicates are never stored. They are worked out from the facts each time, so they cannot drift out of step with the facts.

**How the agent reasoned.** Road usability is inferred by applying the condition rule to the stored condition. Reachability is inferred by forward chaining: starting from the usable roads out of the current town, the rule `Reachable(x, y) ∧ CanUse(y, z) ⇒ Reachable(x, z)` is applied until nothing new can be inferred. The lowest-cost rule is carried out by Uniform Cost Search.

**How queries were answered.** Some questions are answered by looking up a stored fact (distances, location, destination). Others need inference (is a road open, which roads are unavailable, can Kisumu be reached). The notebook labels each answer with how it was obtained. Unknown towns, roads and conditions get an explicit "unknown" answer, and invalid updates are rejected so the KB stays consistent.

**How UCS was integrated.** The Practical Exercise 3 UCS (priority-queue frontier, `bestCosts`, `predecessors`, stale-entry skipping, goal test on removal, search tree record) is reused with only a `showTrace` switch added. It is no longer given a fixed graph. `buildAvailableRoadGraph()` builds its graph from the KB, keeping only the roads the `CanUse` rule allows. With every road open this graph is identical to the Practical Exercise 3 graph, and the agent reproduces the 357 km route. The all-simple-routes check from Practical Exercise 3 confirms it.

**How the agent responded to change.** When told that Nakuru–Mau Summit was closed, the agent updated one pair of facts, inferred the road was unusable, rebuilt its graph without that road and replanned to the 383 km route via Narok, Bomet and Kericho. It also explained which segment of the old plan was blocked and the 26 km cost of the detour. Running the same UCS on the unchanged Practical Exercise 3 graph still recommended the closed road, which shows the KB is what makes the difference. Reopening the road restored the 357 km route. Flooding both roads into Kisumu made the agent conclude, before searching, that it should not move. In the optional extension, the trailer learned of the closure only on reaching Nakuru and had to backtrack through Nairobi.

**Challenges.**
- Deciding what to store and what to infer. Storing `RoadOpen` and `RoadClosed` alongside `RoadCondition` would have been simpler to query but could contradict itself, so they are derived instead.
- Reusing the Practical Exercise 3 UCS without rewriting it. This came down to making the KB produce exactly the graph format that UCS already expected.
- Keeping names consistent (`"mau summit"` vs `"Mau Summit"`) so that facts and queries refer to the same constants.

**Limitations.**
- The reasoning is hand-written Python functions, not a general logic inference engine. A new kind of rule means writing a new function.
- The world is simplified: town-level only, conditions apply to whole roads in both directions, distances are fixed approximations, and the network contains only 10 towns.
- Road conditions are treated as certain. A real trailer would receive uncertain or conflicting reports, but this agent has no notion of confidence or of how old an observation is.
- The agent optimises distance only. Travel time, fuel, tolls and road quality are not considered, and "usable with caution" conditions such as `Wet` carry no extra cost.
- Observations are supplied by the notebook rather than by real sensors or traffic feeds.

**Possible improvements.**
- Give conditions a cost effect (e.g. `Wet` or `Under Repair` adds a delay) so that UCS can prefer drier or faster roads, not just shorter ones.
- Add trailer properties (weight, height) and road limits as facts, so that suitability is inferred from the trailer and the road instead of being a fixed condition.
- Allow one-direction closures and time-limited conditions that expire.
- Store how old and how reliable each observation is, and reason about uncertain information.
- Use a heuristic search such as A* (e.g. straight-line distance to Kisumu) for larger networks.
- Connect observations to a live traffic or weather source.